# Proposed optimizer — 4D Rosenbrock (stagnation-triggered AL)

Synced with `BLACK_BOX_OPTIMIZATION/.../PROPOSED_ALGORITHM/main.ipynb`:

1. Define a black-box loss in **physical** coordinates  
2. Set **4D** box bounds `[-2, 2]^4`  
3. Optionally provide a user initial DoE  
4. Run with **stagnation-triggered** active learning (`stagnation_m`)  
5. Inspect the result

**Acquisition policy:** Cuckoo by default; after `stagnation_m` successive
non-improving Cuckoo steps (exploitation), one active-learning step runs for exploration, then Cuckoo resumes.

## Install (once)

```bash
cd /path/to/proposed_bbopt
python3 -m venv .venv && source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
pip install -e .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from proposed_bbopt import ProposedOptimizer, Hyperparameters

## 1. Loss (4D Rosenbrock)

Minimum at `x* = (1,1,1,1)` with `f = 0`.

In [ ]:
def rosenbrock(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return float(
        np.sum(100.0 * (x[1:] - x[:-1] ** 2) ** 2 + (1.0 - x[:-1]) ** 2)
    )

## 2. Bounds

In [ ]:
vec_lower = np.array([-2.0, -2.0, -2.0, -2.0])
vec_upp = np.array([2.0, 2.0, 2.0, 2.0])
bound = [(float(vec_lower[i]), float(vec_upp[i])) for i in range(len(vec_lower))]
n_dim = len(vec_lower)
print("dimension n =", n_dim)
print("bound:", bound)

## 3. Optional initialization: 'initialization' contains initial points chosen for the algorithm to start. In the below example we choose 5 points.

In [ ]:
USE_CUSTOM_INIT = False

if USE_CUSTOM_INIT:
    initialization = np.array(
        [
            [-1.5, -1.0, 0.0, 0.5],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 1.0, 1.0, 1.0],
            [1.5, -0.5, 1.2, -1.0],
            [-0.5, 1.5, -1.5, 1.8],
        ],
        dtype=float,
    )
else:
    initialization = None

print("initialization:", None if initialization is None else initialization.shape)

## 4. Key Hyperparameters

| Option | Default | Meaning |
|---|---|---|
| `vec_lower` | `[-2,-2,-2,-2]` | Physical lower bound per design variable |
| `vec_upp` | `[2,2,2,2]` | Physical upper bound per design variable |
| `N_t_initial` | 5 | Initial labeled DoE size (true evaluations at start) |
| `N_t_total` | 100 | Total black-box evaluations (DoE + acquired points) |
| `stagnation_m` | 3 | Non-improving Cuckoo steps before one active-learning kick |
| `iter` | 2000 | Surrogate training epochs per outer iteration |
| `L` | 4 | Initial number of network layers (grows if validation stalls) |
| `Nh` | 20 | Hidden-layer width of the surrogate |
| `asymp_N` | 200 | Monte Carlo samples for the AL sensitivity Gram estimate |
| `grid_opt` | 100 | Random grid searches inside active learning |
| `alpha` | 0.01 | Surrogate AdamW learning rate |
| `back_track_rate` | 0.1 | Parameter epsilon in topological derivative approach |


In [ ]:
hyperp = Hyperparameters(
    vec_lower=vec_lower,
    vec_upp=vec_upp,
    N_t_initial=5,
    N_t_total=100,
    stagnation_m=3,
    iter=2000,
    L=4,
    Nh=20,
    asymp_N=200,
    grid_opt=100,
    alpha=0.01,
    back_track_rate=0.1,
)

verbose = True # Printing the progress of the algorithm
seed = 0 # Fixing a seed for reproducibility
workdir = None

print("I:", hyperp.I, ", N_t_total:", hyperp.N_t_total, ", stagnation_m:", hyperp.stagnation_m)

## 5. Run

In [ ]:
optimizer = ProposedOptimizer(
    fitness=rosenbrock,
    bound=bound,
    initialization=initialization,
    hyperp=hyperp,
    verbose=verbose,
    seed=seed,
    workdir=workdir,
)
result = optimizer.execute()

print("best design (physical):", result.best)
print("best loss:", result.best_fitness)
print("total evaluations:", len(result.y))

## 6. Inspect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(result.history_fitness, marker="o", ms=3)
axes[0].set_xlabel("outer iteration")
axes[0].set_ylabel("best loss")
axes[0].set_title("Convergence")
axes[0].set_yscale("log")

axes[1].scatter(result.X[:, 0], result.X[:, 1], c=result.y, s=30, cmap="viridis")
axes[1].scatter(result.best[0], result.best[1], marker="*", s=160, c="C1", label="best")
axes[1].set_xlabel("x0")
axes[1].set_ylabel("x1")
axes[1].set_title("Designs (x0–x1 projection)")
axes[1].legend()

fig.tight_layout()
plt.show()

print("history_fitness (last 5):", result.history_fitness[-5:])